# Load, select and inspect recording streams

Load selected recordings with `DataStructure`, select trial rows, then use each row’s recording and time bounds to select pose and events. Optional pose processing runs separately on each full recording.


In [ ]:
# ===============================================================================
# 1 | Locate This Checkout and Refresh Project Imports After Source Changes
# ===============================================================================

from pathlib import Path
from runpy import run_path

# --- 1.1 | Locate src/ from the checkout root or any notebook subdirectory -------
source_root = next((
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "src")
    if (candidate / "data_conduit").is_dir() and (candidate / "movement_figures").is_dir()
), None)
if source_root is None:
    raise FileNotFoundError("Open this notebook inside the data-conduit checkout.")

# --- 1.2 | Refresh cached project modules before importing the refactored API ----
# Load the bootstrap by file path: even a cached older package cannot shadow it.
# Rerun the following cells afterwards to rebuild data and figures consistently.
refresh_project_imports = run_path(
    str(source_root / "movement_figures" / "notebook_setup.py")
)["refresh_project_imports"]
source_root = refresh_project_imports(source_root)

# --- 1.3 | Import the analysis and plotting functions from this checkout ---------

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from data_conduit.datastructures import slice_stream, slice_stream_for_trial
from data_conduit.integrations.DLC.pose import pose_to_movement
from data_conduit.refactor_qc.training_filter import filter_trials, training_spec
from movement_figures.data_template.loading import build_datastructure, prepare_pose
from movement_figures.video import read_session_video_frame


## Choose recording and trial selection

Use the layout and directories of your actual recordings. `TRIAL_RANGE` refers to each recording’s original trial numbers. `TRAINING_RULES = training_spec()` enables the existing `refactor_qc` selection rules. Its LED exclusions apply only where the rule requests them; `EXCLUDE_LED_ON` provides an additional explicit trial filter. Neither option reconstructs persistent hardware state.

In [ ]:
# === 1| Select Mice, Days and Recording Folders Before Reading Files =============

ROOT = Path('/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training')             # Replace with the directory containing your actual recordings.
LEVEL_NAMES = ('mouseID', 'day')                         # Use () if ROOT directly contains recording folders.
LEVEL_SELECTORS = {'l0_selector': 'MR_M01569515', 'l1_selector': 'Day21'}                                    # Example choices belong here: l0_selector and l1_selector.
INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                 # Optional list of recording folder names; None keeps selected folders.
STREAMS = ('trials', 'events', 'video', 'dlc')             # These figures require events/trials and aligned pose; other sources remain available.

# === 2| Select Trial Rows Before Passing Data to Plotting Functions ==============

TRIAL_RANGE = (1, 8)                                    # Inclusive original trial numbers, applied independently to every recording.
TRAINING_RULES = None                                   # Set training_spec() to reuse refactor_qc's session/min-trial/LED rules.
EXCLUDE_LED_ON = False                                  # True additionally removes all ON-labelled trial rows from the selection.

# === 3| Choose Optional Pose Processing Explicitly ==============================

CONFIDENCE_THRESHOLD = None                             # None preserves raw coordinates; set a threshold after checking DLC likelihoods.
MAX_GAP_FRAMES = None                                   # None keeps tracking gaps; a positive integer fills only short internal runs.
SMOOTHING_WINDOW = None                                 # None disables smoothing; otherwise use an odd frame count of at least three.
READ_BACKGROUND = False                                 # True decodes the first video frame for each selected recording.

## Preview, then read

`select()` reads directory structure; `load()` performs a fresh selection and reads the sources. The preview helps inspect identities, including recordings whose folder names are duplicated under different mice or days. No one-recording restriction is imposed.

In [ ]:
# === 1| Build the Existing DataStructure and Preview the Selected Directories =====

datastructure = build_datastructure(
    ROOT,
    level_names=LEVEL_NAMES,
    streams=STREAMS,
    include=INCLUDE_SESSIONS,
    **LEVEL_SELECTORS,
)

selected_sessions = datastructure.select()              # Directory selection only; no experimental data have been read yet.
display(pd.DataFrame([
    {'session': key, 'path': str(ref.path), **ref.levels}
    for key, ref in selected_sessions.items()
]))

In [ ]:
# === 1| Read the Selected Recordings and Inspect the Plain Trial Table ===========

streams = datastructure.load()                          # Returns the existing StreamMap, a mapping of pandas/xarray objects.
trials = streams['trials']
display(trials.head())

# === 2| Optionally Reuse the Existing Training Session and Trial Rules ============

selected_trials = trials.copy()
if TRAINING_RULES is not None:
    folder_names = {key: ref.path.name for key, ref in datastructure.sessions.items()}
    selected_trials['session_name'] = selected_trials['session'].map(folder_names)
    rules = TRAINING_RULES.rename(columns={'session': 'session_name'})
    selected_trials = filter_trials(
        selected_trials,
        rules,
        session_column='session_name',                  # Spreadsheet folder names are distinct from collision-aware stream identities.
        report=False,
    )                                                   # Keeps the stable session column for subsequent pose/event selection.

# === 3| Apply the Requested Original Trial Range Within Every Recording ==========

selected_trials = slice_stream(
    selected_trials,
    selectors={'trial_index': slice(*TRIAL_RANGE)},                # Each recording numbers its own trials; this does not select global row positions.
)
if EXCLUDE_LED_ON:
    selected_trials = slice_stream(selected_trials, where=lambda rows: rows['LED'].ne('ON'))

display(selected_trials)

In [ ]:
# === 1| Convert and Process Full Recordings Separately ===========================

raw_pose_by_session = {}
pose_by_session = {}
backgrounds = {}

for session_id in selected_trials['session'].drop_duplicates():
    if 'dlc:position' not in streams or 'dlc:confidence' not in streams:
        print('No DLC pose was loaded; trial/event tables remain accessible.')
        break

    position = slice_stream(streams['dlc:position'], selectors={'session': session_id})
    confidence = slice_stream(streams['dlc:confidence'], selectors={'session': session_id})
    if position.sizes['Time'] == 0:
        print(f'{session_id}: no DLC pose available; trial data remain accessible.')
        continue

    raw_pose = pose_to_movement(position, confidence)     # Existing converter; acquired seconds and recording identity are retained.
    pose = prepare_pose(
        raw_pose,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        max_gap_frames=MAX_GAP_FRAMES,
        smoothing_window=SMOOTHING_WINDOW,
    )                                                   # Full-recording processing avoids artificial edges at selected trial boundaries.

    raw_pose_by_session[session_id] = raw_pose
    pose_by_session[session_id] = pose
    if READ_BACKGROUND:
        frame, video_path = read_session_video_frame(datastructure.sessions[session_id].path)
        backgrounds[session_id] = (frame, video_path)   # Each background belongs to this recording's camera coordinates.

# === 2| Compare Missing Coordinates Without Modifying Either Dataset =============

display(pd.DataFrame([
    {
        'session': key,
        'raw_missing': int(raw_pose_by_session[key].position.isnull().sum()),
        'processed_missing': int(pose.position.isnull().sum()),
    }
    for key, pose in pose_by_session.items()
]))

## Select each trial’s pose and events

The recording key selects the source, while the row supplies its time bounds and endpoint inclusion. Event identifiers additionally distinguish raw events sharing a timestamp. Selected rows do not automatically mutate other streams; the following calls make that relationship explicit.

In [ ]:
# === 1| Let Each Selected Trial Supply Its Recording and Time Bounds =============

pose_by_trial = {}
events_by_trial = {}
for _, trial in selected_trials.iterrows():
    key = (trial['session'], int(trial['trial_index']))

    events_by_trial[key] = slice_stream_for_trial(
        streams['events'],
        trial,
    )                                                   # Stable event_index bounds preserve the exact rows when timestamps tie.

    if trial['session'] not in pose_by_session:
        continue

    pose_by_trial[key] = slice_stream_for_trial(
        pose_by_session[trial['session']],
        trial,
        time_coord='time',                              # Movement uses lowercase time; data-conduit's raw arrays use Time.
    )

# Each dictionary entry is ready for a plot function with explicit data inputs.
# Keeping entries separate prevents lines from joining disjoint selected trials.

## Other selections and edits

Direct selection needs no trial table or session argument when the object is already the intended source. `assign` and `drop` are ordinary pandas operations. Relative windows below use the first/last logged event, not an inferred acquisition start/end. Missing recording intervals are not reconstructed.

Video reading remains in `movement_figures.video`. The default `UndistortedVideoData` folder name does not verify lens correction; the image and tracking coordinates must refer to the same video geometry.

In [ ]:
# === 1| Select Arbitrary Coordinates Without a Trial Table =======================

x_positions = None
if 'dlc:position' in streams:
    x_positions = slice_stream(streams['dlc:position'], selectors={'space': 'x'})
short_trials = slice_stream(trials, where=lambda rows: rows['end_time'] - rows['start_time'] < 10)

# === 2| Use Ordinary pandas/xarray Methods to Add or Remove Values ================

trials_with_duration = trials.assign(duration=trials['end_time'] - trials['start_time'])
trials_without_duration = trials_with_duration.drop(columns='duration')

# === 3| Derive Relative Windows From Each Recording's Own Observed Clock ===========

first_minute_events = {}
last_twenty_seconds = {}
for session_id in selected_trials['session'].drop_duplicates():
    recording_events = slice_stream(streams['events'], selectors={'session': session_id})
    if recording_events.empty:
        continue

    start = float(recording_events.index.get_level_values('Time').min())
    end = float(recording_events.index.get_level_values('Time').max())
    first_minute_events[session_id] = slice_stream(recording_events, start=start, end=start + 60)
    last_twenty_seconds[session_id] = slice_stream(recording_events, start=end - 20, end=end)